In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use one GPU; avoids multi-GPU problems if you picked "T4 x2"
!pip install -q chemprop

import pandas as pd, numpy as np, subprocess, shutil, glob, torch
pd.set_option("display.width", 200)
print("GPU available:", torch.cuda.is_available())

data_file = glob.glob("/kaggle/input/**/my_data_clean.csv", recursive=True)[0]   # finds your uploaded file
clean = pd.read_csv(data_file)
print(data_file, clean.shape)

GPU available: True
/kaggle/input/datasets/sruthiakella/nir-thesis-data/my_data_clean.csv (17850, 9)


In [2]:
def make_split(df, seed):
    groups = df.groupby("scaffold").size().sample(frac=1, random_state=seed)
    big = groups[groups > len(df) * 0.005].index.tolist()
    small = [g for g in groups.index if g not in set(big)]
    split, count, n = {}, 0, len(df)
    for g in big + small:
        split[g] = "train" if count < 0.8 * n else ("val" if count < 0.9 * n else "test")
        count += groups[g]
    return df.scaffold.map(split)

SEEDS = [42, 0, 1, 2, 3]
RED_NM = 600
work = "/tmp/ft_work"                    # scratch space, discarded after the run
out = "/kaggle/working/finetune"         # everything here is saved as the run's output
os.makedirs(work, exist_ok=True); os.makedirs(out, exist_ok=True)

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError("command failed: " + cmd[:80])

COMMON = "--smiles-columns chrom solv --target-columns emi_eV --task-type regression --batch-size 64"

In [3]:
for s in SEEDS:
    if os.path.exists(f"{out}/preds_red_only_seed{s}.csv"):
        print("already done: seed", s)
        continue
    d = clean.assign(split=make_split(clean, s))
    d[["chrom", "solv", "emi_eV", "split"]].to_csv(f"{work}/all.csv", index=False)
    # red/NIR training + validation data; the test set stays complete
    red = d[(d.split.isin(["train", "val"]) & (d.emi_nm >= RED_NM)) | (d.split == "test")]
    red[["chrom", "solv", "emi_eV", "split"]].to_csv(f"{work}/red.csv", index=False)
    d[d.split == "test"][["chrom", "solv", "emi_eV"]].to_csv(f"{work}/test.csv", index=False)
    print(f"seed {s}: red train {(red.split == 'train').sum()}, red val {(red.split == 'val').sum()}")

    # 1) general model, all training data (same settings as notebook 04)
    run(f"rm -rf {work}/m_base")
    run(f"chemprop train --data-path {work}/all.csv --splits-column split {COMMON} --epochs 40 --patience 10 "
        f"--depth 4 --message-hidden-dim 300 --ffn-num-layers 2 --pytorch-seed {s} --save-dir {work}/m_base")

    # 2) fine-tune that model on red/NIR molecules with a 10x smaller learning rate
    run(f"rm -rf {work}/m_ft")
    run(f"chemprop train --data-path {work}/red.csv --splits-column split {COMMON} "
        f"--checkpoint {work}/m_base/model_0/best.pt "
        f"--epochs 30 --patience 10 --warmup-epochs 2 --init-lr 1e-5 --max-lr 1e-4 --final-lr 1e-5 "
        f"--pytorch-seed {s} --save-dir {work}/m_ft")

    # 3) red-only model from scratch (comparison)
    run(f"rm -rf {work}/m_red")
    run(f"chemprop train --data-path {work}/red.csv --splits-column split {COMMON} --epochs 40 --patience 10 "
        f"--depth 4 --message-hidden-dim 300 --ffn-num-layers 2 --pytorch-seed {s} --save-dir {work}/m_red")

    for tag, m in [("base", "m_base"), ("ft", "m_ft"), ("red_only", "m_red")]:
        run(f"chemprop predict --test-path {work}/test.csv --smiles-columns chrom solv "
            f"--model-paths {work}/{m} --preds-path {out}/preds_{tag}_seed{s}.csv")
    # keep the general and fine-tuned models: you'll reuse them for screening
    for tag, m in [("base", "m_base"), ("ft", "m_ft")]:
        shutil.copy(f"{work}/{m}/model_0/best.pt", f"{out}/{tag}_seed{s}.pt")
    print("seed", s, "done")

seed 42: red train 2148, red val 342
seed 42 done
seed 0: red train 2172, red val 369
seed 0 done
seed 1: red train 2300, red val 340
seed 1 done
seed 2: red train 2172, red val 306
seed 2 done
seed 3: red train 2177, red val 340
seed 3 done


In [4]:
def metrics(true, pred):
    err = pred - true
    hit = pred >= 700
    r = {"MAE all": np.abs(err).mean()}
    for lab, lo, hi in [("<500", 0, 500), ("500-650", 500, 650), ("650-700", 650, 700), (">=700", 700, 2000)]:
        m = (true >= lo) & (true < hi)
        r[f"MAE {lab}"] = np.abs(err[m]).mean()
    r["bias >=700"] = err[true >= 700].mean()
    r["recall %"] = 100 * hit[true >= 700].mean()          # share of real NIR emitters found
    r["false hits"] = int((hit & (true < 650)).sum())      # truly < 650 nm but predicted NIR
    r["precision %"] = 100 * (true[hit] >= 700).mean() if hit.any() else np.nan
    return r

rows = []
for s in SEEDS:
    true = clean[make_split(clean, s) == "test"].emi_nm.values
    P = {t: 1239.84 / pd.read_csv(f"{out}/preds_{t}_seed{s}.csv").iloc[:, -1].values   # last column = predictions
         for t in ["base", "ft", "red_only"]}
    P["two-stage"] = np.where(P["base"] >= RED_NM, P["ft"], P["base"])
    for t, p in P.items():
        rows.append({"seed": s, "approach": t, **metrics(true, p)})

res = pd.DataFrame(rows)
res.round(1).to_csv(f"{out}/finetune_results.csv", index=False)
cols = ["MAE all", "MAE <500", "MAE 650-700", "MAE >=700", "bias >=700", "recall %", "false hits", "precision %"]
print(res.groupby("approach", sort=False)[cols].agg(["mean", "std"]).round(1).to_string())

          MAE all       MAE <500       MAE 650-700      MAE >=700       bias >=700       recall %       false hits       precision %      
             mean   std     mean   std        mean  std      mean   std       mean   std     mean   std       mean   std        mean   std
approach                                                                                                                                  
base         33.2   3.0     27.1   1.6        39.8  5.8      51.5  17.2      -33.6  22.8     63.6  13.0        7.8   8.8        67.2  10.8
ft           87.4  14.0    118.5  23.8        25.6  3.7      42.1   9.5      -28.1  13.7     63.2  10.4        5.8   8.1        75.2  12.2
red_only    135.2   9.3    191.1   9.1        26.1  5.0      48.8   7.4      -39.8  12.8     54.7   9.2       11.4  12.2        64.0  18.4
two-stage    33.6   3.2     27.2   1.7        33.7  6.7      46.4  12.3      -32.4  17.3     63.2  10.4        5.6   8.1        75.4  12.1


In [5]:
from scipy import stats
for col in ["MAE all", "MAE >=700", "precision %"]:
    b = res[res.approach == "base"].set_index("seed")[col]
    t = res[res.approach == "two-stage"].set_index("seed")[col]
    diff = t - b
    print(f"{col}: two-stage minus general = {diff.mean():+.1f} ± {diff.std():.1f}, "
          f"better in {int(((diff < 0) if 'MAE' in col else (diff > 0)).sum())}/5 splits, "
          f"paired t-test p = {stats.ttest_rel(t, b).pvalue:.3f}")

MAE all: two-stage minus general = +0.4 ± 0.9, better in 2/5 splits, paired t-test p = 0.375
MAE >=700: two-stage minus general = -5.1 ± 12.3, better in 3/5 splits, paired t-test p = 0.411
precision %: two-stage minus general = +8.2 ± 18.7, better in 3/5 splits, paired t-test p = 0.385


In [6]:
!cd /kaggle/working && zip -qr finetune_results.zip finetune && ls -lh finetune_results.zip

-rw-r--r-- 1 root root 27M Oct 23 17:09 finetune_results.zip
